# ResNet18 vs ViT-B/16: Comparison Table + Plots
Merges your ResNet18 results with Arjot's ViT-B/16 results into one comparison table and the core accuracy-vs-data-fraction chart. Confirmed final numbers are hardcoded below since they were gathered from separate notebooks/sessions; update if either number changes.

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

# Confirmed test accuracy (top-1) at each data fraction, from both notebooks
results = {
    'data_fraction': [10, 50, 100],
    'resnet18_test_acc': [0.619, 0.696, 0.746],
    'vit_b16_test_acc': [0.727, 0.794, 0.821],
}

df = pd.DataFrame(results)
df['gap'] = df['vit_b16_test_acc'] - df['resnet18_test_acc']
print(df)

## Comparison table (formatted, for the report)

In [ ]:
table = df.copy()
table['resnet18_test_acc'] = (table['resnet18_test_acc'] * 100).round(1).astype(str) + '%'
table['vit_b16_test_acc'] = (table['vit_b16_test_acc'] * 100).round(1).astype(str) + '%'
table['gap'] = (df['gap'] * 100).round(1).astype(str) + ' pts'
table.columns = ['Data Fraction (%)', 'ResNet18 Test Acc', 'ViT-B/16 Test Acc', 'ViT Advantage']
display(table)

## Chart: accuracy vs. data fraction (both models)
Core comparison figure for the report.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))

ax.plot(df['data_fraction'], df['resnet18_test_acc'] * 100, marker='o', linewidth=2, label='ResNet18', color='#1f77b4')
ax.plot(df['data_fraction'], df['vit_b16_test_acc'] * 100, marker='s', linewidth=2, label='ViT-B/16', color='#d62728')

for x, y in zip(df['data_fraction'], df['resnet18_test_acc'] * 100):
    ax.annotate(f'{y:.1f}%', (x, y), textcoords='offset points', xytext=(0, -15), ha='center', fontsize=9)
for x, y in zip(df['data_fraction'], df['vit_b16_test_acc'] * 100):
    ax.annotate(f'{y:.1f}%', (x, y), textcoords='offset points', xytext=(0, 10), ha='center', fontsize=9)

ax.set_xlabel('Training Data Used (%)', fontsize=11)
ax.set_ylabel('Test Accuracy (Top-1, %)', fontsize=11)
ax.set_title('ResNet18 vs ViT-B/16: Test Accuracy by Data Fraction', fontsize=12)
ax.set_xticks(df['data_fraction'])
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(f'{PROJECT_DIR}/results/accuracy_vs_fraction.png', dpi=200, bbox_inches='tight')
plt.show()

## Loss/accuracy curves per model (overfitting visualization)
Uses per-epoch history. ResNet18 history comes from your own training run outputs (paste below if not already saved to a file). ViT history is loaded directly from Arjot's saved `*_history.json` files (already in this format from his zip export).

In [ ]:
import json

# Load ViT per-epoch history for the 100% run (best config: lr=5e-5, wd=0.0001)
# Update this path to wherever the history JSON lives (from his results zip or your Drive shortcut)
vit_history_path = f'{PROJECT_DIR}/results/vit_b_16_frac100_lr5e-05_wd0.0001_history.json'

with open(vit_history_path) as f:
    vit_history_100 = json.load(f)

vit_epochs = [e['epoch'] for e in vit_history_100]
vit_train_acc = [e['train_top1'] * 100 for e in vit_history_100]
vit_val_acc = [e['val_top1'] * 100 for e in vit_history_100]

In [ ]:
# ResNet18 per-epoch history at 100% data, from your B2 sweep run.
# Fill these in from your printed training output (Epoch XX/10 | train ... acc ... | val ... acc ...)
# for the fraction='100' run. Replace with your actual per-epoch numbers.
resnet18_epochs = list(range(1, 11))
resnet18_train_acc = []  # e.g. [0.55, 0.68, 0.75, ...]
resnet18_val_acc = []    # e.g. [0.52, 0.60, 0.65, ...]

if not resnet18_train_acc or not resnet18_val_acc:
    print('Fill in resnet18_train_acc / resnet18_val_acc from your saved epoch printouts before running the next cell.')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

if resnet18_train_acc and resnet18_val_acc:
    axes[0].plot(resnet18_epochs, resnet18_train_acc, marker='o', label='Train', color='#1f77b4')
    axes[0].plot(resnet18_epochs, resnet18_val_acc, marker='o', label='Val', color='#ff7f0e')
    axes[0].set_title('ResNet18 (100% data): Train vs Val Accuracy')
    axes[0].set_xlabel('Epoch')
    axes[0].set_ylabel('Accuracy (%)')
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)
else:
    axes[0].text(0.5, 0.5, 'ResNet18 epoch data not filled in yet', ha='center', va='center', transform=axes[0].transAxes)

axes[1].plot(vit_epochs, vit_train_acc, marker='s', label='Train', color='#1f77b4')
axes[1].plot(vit_epochs, vit_val_acc, marker='s', label='Val', color='#d62728')
axes[1].set_title('ViT-B/16 (100% data): Train vs Val Accuracy')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy (%)')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(f'{PROJECT_DIR}/results/overfitting_comparison.png', dpi=200, bbox_inches='tight')
plt.show()

## Next steps
- Fill in `resnet18_train_acc`/`resnet18_val_acc` above from your saved B2 sweep output, then rerun the last two cells for the full overfitting comparison figure
- Best combo overall: **ViT-B/16 at 100% data (82.1% test acc)** — use this for the augmentation-off ablation
- Both charts are saved to `{PROJECT_DIR}/results/` as PNGs, ready to drop into the report